# **＜重要＞本章で使用するファイル**

- customer_data.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

# Menu.94 PythonからOpenAI APIを使おう

必要なライブラリをインストールする

In [ ]:
!pip install openai langgraph langchain-openai langchain-core pandas

クライアントの作成、モデル名の入力をし、リクエストを送信する

In [ ]:
from openai import OpenAI

# APIキーを設定（実際のキーに置き換えてください）
API_KEY = "sk-proj-XXXXX..."
client = OpenAI(api_key=API_KEY)

# Responses APIでメッセージを送信
response = client.responses.create(
  model="gpt-5.4-mini",
  input="日本の首都はどこですか？"
)

# 結果を表示
print(response.output_text)

モデル名やレスポンスIDを確認する

In [ ]:
# 回答テキスト
print(response.output_text)

# モデル名
print(response.model) # 例: "gpt-5.4-mini-2026-03-17"

# レスポンスID（デバッグやログに便利）
print(response.id) # 例: "resp_01cb2e1ed318..."

# Menu.95 AIエージェントとLangGraphの基礎を理解しよう

質問とLLMの回答を保持する

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

ノードを定義する

In [ ]:
from langchain_openai import ChatOpenAI

# LLMを準備
API_KEY = "sk-proj-XXXXX..."
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, api_key=API_KEY)

# ノード関数を定義
def chat_node(state: AgentState) -> dict:
  """LLMにメッセージを送り、返答を受け取る"""
  # 状態からメッセージを取り出してLLMに渡す
  response = llm.invoke(state["messages"])
  return {"messages": [response]} # LLMの返答を状態に追加

グラフを作成し、ノードとエッジを追加する

In [ ]:
from langgraph.graph import StateGraph, START, END
# グラフを作成（どんな状態を使うか指定）

workflow = StateGraph(AgentState)

# ノードを追加（名前と関数を指定）
workflow.add_node("chat", chat_node)

# エッジを追加（どの順番で処理するか指定）
workflow.add_edge(START, "chat") # 開始 → chat
workflow.add_edge("chat", END) # chat → 終了

コンパイルをして、アプリケーションを実行する

In [ ]:
# コンパイル
app = workflow.compile()

# 実行
from langchain_core.messages import HumanMessage
result = app.invoke({
  "messages": [HumanMessage(content="日本で一番高い山は？")]
})

print(result["messages"][-1].content)

# Menu.96 エージェント用のツールを作ろう

3つのツールを実装する

In [ ]:
import pandas as pd
from langchain_core.tools import tool

@tool
def load_csv_head(file_path: str, n: int = 5) -> dict:
  """CSVを読み込み、shape/columns/head(先頭n行)を返します。

  Args:
    file_path: CSVファイルのパス
    n: 表示する行数（1~10、デフォルト5）
  """
  try:
    df = pd.read_csv(file_path)
    n = max(1, min(int(n), 10)) # 1~10に制限
    return {
      "success": True,
      "shape": list(df.shape),
      "columns": df.columns.tolist(),
      "head": df.head(n).to_dict(orient="records"),
    }
  except Exception as e:
    return {"success": False, "error": str(e)}

@tool
def group_by_mean(file_path: str, group_col: str, value_col: str) -> dict:
  """指定した列でグループ化し、別の列の平均を計算します。

  Args:
    file_path: CSVファイルのパス
    group_col: グループ化に使う列名
    value_col: 平均を計算する列名（数値列を指定してください）
  """
  try:
    df = pd.read_csv(file_path)
    for col in (group_col, value_col):
      if col not in df.columns:
        return {"success": False, "error": f"列が見つかりません: {col}"}

    result = df.groupby(group_col)[value_col].mean().reset_index()
    return {
      "success": True,
      "table": result.to_dict(orient="records"),
    }
  except Exception as e:
    return {"success": False, "error": str(e)}

@tool
def compute_corr(file_path: str, x_col: str, y_col: str) -> dict:
  """2つの数値列の相関係数を計算します。
  Args:
    file_path: CSVファイルのパス
    x_col: 1つ目の列名
    y_col: 2つ目の列名
  """
  try:
    df = pd.read_csv(file_path)
    for col in (x_col, y_col):
      if col not in df.columns:
        return {"success": False, "error": f"列が見つかりません: {col}"}

    corr = df[x_col].corr(df[y_col])
    return {
      "success": True,
      "correlation": round(corr, 4),
    }
  except Exception as e:
    return {"success": False, "error": str(e)}

3つのツールの動作を確認する

In [ ]:
# CSV概要を確認
result = load_csv_head.invoke({"file_path": "customer_data.csv", "n": 3})
print("=== load_csv_head ===")
print("shape:", result["shape"])
print("columns:", result["columns"])

# グループ別平均
result = group_by_mean.invoke({
  "file_path": "customer_data.csv",
  "group_col": "favorite_category",
  "value_col": "total_amount"
})
print()
print("=== group_by_mean ===")
for row in result["table"]:
  print(row)

# 相関係数
result = compute_corr.invoke({
  "file_path": "customer_data.csv",
  "x_col": "age",
  "y_col": "total_amount"
})
print()
print("=== compute_corr ===")
print("相関係数:", result["correlation"])

# Menu.97 ツールをエージェントに組み込もう

ツールをLLMに紐付ける

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, api_key=API_KEY)

tools = [load_csv_head, group_by_mean, compute_corr]
llm_with_tools = llm.bind_tools(tools)

エージェントを実装する

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

API_KEY = "sk-proj-XXXXX..."

# 状態を定義：会話履歴を保持する
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

# LLMとツールを準備
tools = [load_csv_head, group_by_mean, compute_corr]
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, api_key=API_KEY)
llm_with_tools = llm.bind_tools(tools)

# システムプロンプト：エージェントの役割を定義
SYSTEM_PROMPT = """あなたはデータ分析エージェントです。
与えられたツールを使って、ユーザーの質問に答えてください。

重要なルール：
- 実行できるのは「提供されたツール呼び出し」だけです
- ツールにはfile_path引数でCSVファイルのパスを渡してください
- まずload_csv_headで列名を確認してから分析を始めてください
- 結果は簡潔にまとめてください
"""

# agent ノード：LLMが考えるターン
def agent_node(state: AgentState) -> dict:
  messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
  response = llm_with_tools.invoke(messages)
  return {"messages": [response]}

# グラフを構築
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node) # LLMが考えるノード
workflow.add_node("tools", ToolNode(tools)) # ツールを実行するノード
workflow.add_edge(START, "agent") # 開始 → agent
workflow.add_conditional_edges("agent", tools_condition)
# tools → agent（結果を見てまた考える）
workflow.add_edge("tools", "agent")

app = workflow.compile()

appに質問して、エージェントを実行する

In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke({
  "messages": [HumanMessage(content="customer_data.csvで、年齢と累計購入金額の相関はありますか？")]
})
print(result["messages"][-1].content)

# Menu.98 構造化出力で計画を生成しよう

LLMの出力形式を設定する

In [ ]:
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI

API_KEY = "sk-proj-XXXXX..."
# 計画の型を定義
class Plan(TypedDict):
  steps: list[str]

# LLMを準備
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, api_key=API_KEY)

# 構造化出力を設定
planner = llm.with_structured_output(Plan)

LLMに分析計画を生成させる

In [ ]:
plan = planner.invoke(
  "customer_data.csvの離脱傾向を分析してレポートを作成してください。"
  "使えるツールは load_csv_head, group_by_mean, compute_corr です。"
  "最大5ステップで計画を立ててください。"
)

for i, step in enumerate(plan["steps"], 1):
  print(i, step)

# Menu.99 計画に沿って分析するエージェントを作ろう

計画を生成して、分析グラフに組み込む

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

API_KEY = "sk-proj-XXXXX..."

# 計画の型
class Plan(TypedDict):
  steps: list[str]

# 状態の定義
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]
  plan: list[str] # 生成された計画

# LLMとツールの準備
tools = [load_csv_head, group_by_mean, compute_corr]
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, api_key=API_KEY)
llm_with_tools = llm.bind_tools(tools)
planner = llm.with_structured_output(Plan)

SYSTEM_PROMPT = """あなたはデータ分析エージェントです。
計画に沿って、ツールを使って分析を進めてください。
ツールにはfile_path引数でCSVファイルのパスを渡してください。
分析が終わったら、結果をまとめてください。
"""

def planning_node(state: AgentState) -> dict:
  """ユーザーの依頼から分析計画を生成する"""
  user_request = state["messages"][-1].content
  plan = planner.invoke(
    f"依頼: {user_request}\n"
    f"使えるツール: load_csv_head, group_by_mean, compute_corr\n"
    f"最大5ステップで計画を立ててください。"
  )
  return {"plan": plan["steps"]}

def agent_node(state: AgentState) -> dict:
  """計画に沿ってツールを使い分析を進める"""
  plan_text = "\n".join([f"- {s}" for s in state.get("plan", [])])
  messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    SystemMessage(content=f"計画:\n{plan_text}"),
  ] + state["messages"]

  response = llm_with_tools.invoke(messages)
  return {"messages": [response]}

# グラフの構築
workflow = StateGraph(AgentState)
workflow.add_node("planning", planning_node)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "planning")
workflow.add_edge("planning", "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

app = workflow.compile()

計画を立てて、顧客傾向を分析する

In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke({
  "messages": [
    HumanMessage(content="customer_data.csvの顧客傾向を分析してください")
  ],
  "plan": []
})

print("=== 生成された計画 ===")
for i, step in enumerate(result["plan"], 1):
    print(i, step)

print()
print("=== 分析結果 ===")
print(result["messages"][-1].content)